# RNN 실습 3: 사인파로 RNN → LSTM · GRU 바꿔 보기

| 순서 | 내용 | 시간 |
|---|---|---|
| 1 | 세 모델의 입력·출력 모양과 파라미터 수 | 5분 |
| 2 | 윈도우 20, 100에서 RNN·LSTM·GRU 비교 | 15분 |
| 3 | 문제 (코드를 직접 고쳐 보기) | 남는 시간·과제 |

1일차 실습 2의 사인파 코드에서 `nn.RNN` 자리만 `nn.LSTM`, `nn.GRU`로 바꿈.
- **LSTM**: 게이트 3개(잊기·입력·출력) + 셀 상태 C
- **GRU**: 게이트 2개(리셋·업데이트)

In [1]:
import time
import torch
import torch.nn as nn

torch.manual_seed(0)

## 1. 입력·출력 모양과 파라미터 수

In [2]:
x = torch.randn(4, 5, 1)                                  # 배치 4, 길이 5, 값 1개
for Cell in [nn.RNN, nn.LSTM, nn.GRU]:
    layer = Cell(input_size=1, hidden_size=8, batch_first=True)
    out, state = layer(x)
    n = sum(p.numel() for p in layer.parameters())
    if Cell is nn.LSTM:                                      # LSTM은 (h_n, c_n) 두 개를 돌려줌
        h_n, c_n = state
        print(f"{Cell.__name__:5s} out {tuple(out.shape)}  h_n {tuple(h_n.shape)}  c_n {tuple(c_n.shape)}  파라미터 {n}")
    else:
        print(f"{Cell.__name__:5s} out {tuple(out.shape)}  h_n {tuple(state.shape)}  파라미터 {n}")

RNN   out (4, 5, 8)  h_n (1, 4, 8)  파라미터 88
LSTM  out (4, 5, 8)  h_n (1, 4, 8)  c_n (1, 4, 8)  파라미터 352
GRU   out (4, 5, 8)  h_n (1, 4, 8)  파라미터 264


out 모양은 셋 다 같음. 파라미터는 RNN 88개 기준 GRU 3배(264), LSTM 4배(352): 게이트마다 W가 따로 있기 때문.

## 2. 윈도우 20, 100에서 비교
`run_sine(모델, 윈도우, hidden=16, layers=1, epochs=20, lr=0.01)` 의 값을 바꾸면 여러 실험을 할 수 있음.

In [3]:
def sine_data(window):
    """사인파 1,000개 점 → 윈도우 → 앞 80% 학습, 뒤 20% 테스트"""
    t = torch.linspace(0, 40 * torch.pi, 1000)
    wave = torch.sin(t)
    X = torch.stack([wave[i:i + window] for i in range(len(wave) - window)]).unsqueeze(-1)
    y = wave[window:].unsqueeze(-1)
    n = int(len(X) * 0.8)
    return X[:n], y[:n], X[n:], y[n:]

class SeqModel(nn.Module):
    def __init__(self, Cell, hidden=16, layers=1):
        super().__init__()
        self.rnn = Cell(input_size=1, hidden_size=hidden, num_layers=layers, batch_first=True)  # nn.RNN / nn.LSTM / nn.GRU
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.rnn(x)            # LSTM도 out은 같은 모양이라 코드 그대로
        return self.fc(out[:, -1, :])

def run_sine(Cell, window, hidden=16, layers=1, epochs=20, lr=0.01):
    X_tr, y_tr, X_te, y_te = sine_data(window)
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    torch.manual_seed(0)
    model = SeqModel(Cell, hidden, layers)
    loss_fn = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    start = time.time()
    for epoch in range(epochs):
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        mse = loss_fn(model(X_te), y_te).item()
    n = sum(p.numel() for p in model.parameters())
    print(f"윈도우 {window:3d} | {Cell.__name__:5s} test MSE {mse:.6f} | 파라미터 {n:5d} | {time.time() - start:.1f}초")
    return model

for window in [20, 100]:
    for Cell in [nn.RNN, nn.LSTM, nn.GRU]:
        run_sine(Cell, window)
    print()

윈도우  20 | RNN   test MSE 0.000008 | 파라미터   321 | 1.5초


윈도우  20 | LSTM  test MSE 0.000002 | 파라미터  1233 | 1.7초


윈도우  20 | GRU   test MSE 0.000006 | 파라미터   929 | 3.5초



윈도우 100 | RNN   test MSE 0.000065 | 파라미터   321 | 5.5초


윈도우 100 | LSTM  test MSE 0.000005 | 파라미터  1233 | 2.0초


윈도우 100 | GRU   test MSE 0.000012 | 파라미터   929 | 14.6초



- 윈도우 20: 사인파는 규칙적이고 짧아서 세 모델 모두 잘 맞힘
- 윈도우 100: RNN은 약 8배 나빠지지만 LSTM·GRU는 거의 그대로 → **순서가 길수록 게이트 구조가 유리** (장기 의존성 문제)
- 파라미터는 GRU 약 3배, LSTM 약 4배. 학습 시간은 실행 환경마다 다름 (PyTorch는 LSTM 계산이 최적화돼 있어 오히려 빠를 때도 있음)

## 3. 문제: 코드를 직접 고쳐 보기
1. 윈도우 50으로 세 모델 비교하기: `for Cell in [...]: run_sine(Cell, 50)`. RNN과 LSTM·GRU의 차이는 20과 100 사이 어디쯤일까?
2. 윈도우 100에서 RNN의 hidden을 32로 늘리면 LSTM을 따라잡을까? `run_sine(nn.RNN, 100, hidden=32)`
3. 윈도우 100에서 RNN을 2층으로 쌓으면? `run_sine(nn.RNN, 100, layers=2)` 층을 쌓는 것으로 장기 의존성 문제가 해결될까?
4. 윈도우 100에서 RNN을 40 에폭 학습하면? `run_sine(nn.RNN, 100, epochs=40)`
5. LSTM이 돌려주는 `h_n`과 `c_n`을 직접 출력해 비교하기. 값의 범위가 어떻게 다를까?
   ```python
   lstm = nn.LSTM(1, 4, batch_first=True)
   out, (h_n, c_n) = lstm(torch.randn(1, 50, 1) * 3)
   print(h_n, c_n)
   ```
6. (도전) `SeqModel`을 고쳐 `nn.LSTM`을 쓸 때만 `c_n`도 받아서 출력해 보기.

<details><summary>정답 보기</summary>

1. 실행 결과 예: RNN 0.00010, LSTM 0.00001, GRU 0.00001. 윈도우 20(RNN 0.000008)보다 RNN만 크게 나빠지고 LSTM·GRU는 그대로. 길어질수록 차이가 커짐.
2. 0.000024로 좋아지지만 LSTM(0.000005)보다는 나쁨. 기억 칸을 늘려도 먼 과거의 기울기가 사라지는 문제는 남음.
3. 0.000081로 거의 그대로. 층을 쌓는 것은 '위로' 깊어지는 것이고, 기울기 소실은 '시간 방향'으로 길어서 생기는 문제라 해결되지 않음.
4. 0.000031로 좋아지지만 여전히 LSTM보다 나쁨. 오래 학습하면 조금 나아지지만 구조의 한계는 남음.
5. h = o × tanh(C)라서 h_n은 항상 −1 ~ 1. c_n(셀 상태)은 덧셈으로 쌓여서 1보다 커질 수 있음 → C가 '장기 기억 통로'라는 뜻.
6. `out, state = self.rnn(x)` 후 `if isinstance(self.rnn, nn.LSTM): h_n, c_n = state` 처럼 나눠 받기.
</details>